In [ ]:
"""
Titanic Data Analysis and JSON Export
Author: [Your Name]
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""
 
import pandas as pd
import numpy as np
import json
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

In [ ]:
df=pd.read_csv(CSV_FILE)
print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

In [ ]:
# Select numeric columns only
numeric_columns = df.select_dtypes(include='number').columns

# Calculate statistics (.mean, median, std)
stats = df[numeric_columns].agg(['mean', 'median', 'std']).T

print("Numeric column statistics:")
print(stats.round(2))

In [ ]:
# Identify numeric columns
numeric_cols = numeric_columns

# Prepare to build a column-wise statistics dictionary in the next cell


In [ ]:
# Create a dictionary to store results
results = {}
for col in numeric_cols:
    results[col] = {
        'mean': df[col].mean(),
        'median': df[col].median(),
        'std': df[col].std()
    }
print("Numeric column statistics:")
print(results) 
# Calculate statistics for each numeric column
for col in numeric_cols:
    mean = df[col].mean()
    median = df[col].median()
    std = df[col].std()
    print(f"{col} - Mean: {mean}, Median: {median}, Std: {std}")
# Store results in the dictionary
    results[col] = {
        'mean': mean,
        'median': median,
        'std': std
    }
print("Numeric column statistics:")
print(results)




In [ ]:
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)
 
missing_data = {}
missing_data_summary = []
# Calculate and display missing value counts and percentages for each column
missing_data_summary = []
for col in df.columns:
    missing_count = df[col].isnull().sum()  # Correctly calculate missing values
    missing_percent = (missing_count / len(df)) * 100  # Calculate percentage
    missing_data_summary.append({
        "Column": col,
        "Missing Count": missing_count,
        "Missing Percentage": f"{missing_percent:.2f}%"
    })

# Convert to DataFrame for a neat presentation
missing_data_df = pd.DataFrame(missing_data_summary)

# Display the results
print("\nMissing Data Summary:")
print(missing_data_df.to_string(index=False))
    

In [ ]:
# Create a copy of the dataframe for feature engineering
df_features = df.copy()
 
# Feature 1: Family Size
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1  # +1 for the individual
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))
 
# Feature 2: Is Alone
df_features['IsAlone'] = (df_features['FamilySize'] == 1).astype(int)
print(df_features[['FamilySize', 'IsAlone']].head(10))
 
# Feature 3: Age Groups
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 30:
        return 'Young Adult'
    elif age < 50:
        return 'Adult'
    else:
        return 'Senior'

# Apply the function to categorize ages
df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))
 
# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)
 
# Example of analyzing Family Size by survival
print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)
 
# Feature differentiation analysis
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)
 
survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]
 
print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

In [ ]:
# Step 3: Create Classes for JSON Export
 
from datetime import datetime

class Passenger:
    """
    Represents a passenger with all their information.
    """
    def __init__(self, passenger_id, name, age, sex, survived, pclass, 
                 fare, embarked=None, family_size=None, is_alone=None, title=None):
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = int(is_alone) if pd.notna(is_alone) else None
        self.title = str(title) if pd.notna(title) else None
    
    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""
        return {
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'title': self.title,
        }
 
class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []  # Will store Passenger objects
        self._create_passengers()
    
    @staticmethod
    def _extract_title(name):
        if pd.isna(name):
            return None

        name = str(name)
        if ',' not in name or '.' not in name:
            return None

        return name.split(',', 1)[1].split('.', 1)[0].strip()
    
    def _create_passengers(self):
        """Create Passenger objects from dataframe."""
        for idx, row in self.dataframe.iterrows():
            passenger = Passenger(
                passenger_id=row.get('PassengerId', idx),
                name=row.get('Name'),
                age=row.get('Age'),
                sex=row.get('Sex'),
                survived=row.get('Survived'),
                pclass=row.get('Pclass'),
                fare=row.get('Fare'),
                embarked=row.get('Embarked'),
                family_size=row.get('FamilySize'),
                is_alone=row.get('IsAlone'),
                title=self._extract_title(row.get('Name')),
            )
            self.passengers.append(passenger)
    
    def to_json(self, filename='titanic_data.json'):
        """Export dataset to JSON file."""
        output_path = Path(filename)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                'total_passengers': len(self.passengers),
                'survival_rate': round(float(self.dataframe['Survived'].mean()), 4),
                'columns': list(self.dataframe.columns),
            },
            'passengers': [p.to_dict() for p in self.passengers]
        }
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        print(f"Data exported to {output_path}")
        return data
    
    def get_summary_stats(self):
        """Get summary statistics."""
        ages = [p.age for p in self.passengers if p.age is not None]
        fares = [p.fare for p in self.passengers if p.fare is not None]
        survived_count = sum(1 for p in self.passengers if p.survived == 1)
        did_not_survive_count = sum(1 for p in self.passengers if p.survived == 0)
        
        return {
            'total_passengers': len(self.passengers),
            'survived': survived_count,
            'did_not_survive': did_not_survive_count,
            'average_age': round(sum(ages) / len(ages), 2) if ages else None,
            'average_fare': round(sum(fares) / len(fares), 2) if fares else None,
        }
 
if 'df_features' in locals() and not df_features.empty:
    dataset = TitanicDataset(df_features)
    print(f"Created dataset with {len(dataset.passengers)} passengers.")

    print("\nSummary statistics:")
    summary_stats = dataset.get_summary_stats()
    for key, value in summary_stats.items():
        print(f"{key}: {value}")

    dataset.to_json(JSON_FILE)

In [ ]:
import json
from pathlib import Path

json_file_path = JSON_FILE if 'JSON_FILE' in locals() else Path('data') / 'titanic_data.json'
print("JSON file:", json_file_path.resolve())

if not json_file_path.exists():
    print("JSON file is missing. Attempting to create it now...")

    if 'df_features' in locals() and not df_features.empty and 'TitanicDataset' in locals():
        dataset = TitanicDataset(df_features)
        dataset.to_json(json_file_path)
    else:
        print("Could not create JSON automatically. Run the feature engineering and export cells first.")

if json_file_path.exists():
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        print("JSON data loaded successfully.")
        print(f"Loaded {len(json_data.get('passengers', []))} passenger records.")
        print(json_data['metadata'])
    except json.JSONDecodeError as e:
        print(f"Failed to decode JSON: {e}")
else:
    print("The specified file was not found. Please check the path and filename.")

